In [101]:
from abc import ABC, abstractmethod

class Runnable(ABC):
    @abstractmethod
    def invoke(input_data):
        pass

In [102]:
import random


class LLM(Runnable):
    def __init__(self):
        print("LLM created")

    def invoke(self, prompt):
        responses = ['This is the first response from llm', 'This is the second response from llm', 'This is the third response from llm']
        return {'response': random.choice(responses)}

    def predict(self, prompt):
        responses = ['This is the first response from llm', 'This is the second response from llm', 'This is the third response from llm']
        return {'warning': 'This method is depricated, use "invoke" instead', 'response': random.choice(responses)}

In [103]:
class PromptTemplate(Runnable):
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def invoke(self, input_dict):
        return self.template.format(**input_dict)
    
    def format(self, input_dict):
        return {'warning': 'This method is depricated, use "invoke" instead', 'response': self.template.format(**input_dict)}

In [104]:
llm = LLM()

LLM created


In [105]:
template = PromptTemplate(
    template="Write a {length} blog on {topic}",
    input_variables=['length', 'topic']
)

In [106]:
prompt_1 = template.format({'length': 'short', 'topic': 'LangChain'})
prompt_1

{'warning': 'This method is depricated, use "invoke" instead',
 'response': 'Write a short blog on LangChain'}

In [107]:
prompt_2 = template.invoke({'length': 'short', 'topic': 'LangChain'})
prompt_2

'Write a short blog on LangChain'

In [108]:
llm.predict(prompt=prompt_1)

{'warning': 'This method is depricated, use "invoke" instead',
 'response': 'This is the second response from llm'}

In [109]:
llm.invoke(prompt=prompt_2)

{'response': 'This is the third response from llm'}

In [110]:
class RunnableChain(Runnable):

    def __init__(self, runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):
        
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)

        return input_data

In [111]:
class StringOutputParser(Runnable):
    def __init__(self):
        pass

    def invoke(self, input_data):
        return input_data['response']

In [112]:
str_parser = StringOutputParser()

In [113]:
chain = RunnableChain([template, llm, str_parser])

In [114]:
chain.invoke({'length': 'short', 'topic': 'LangChain'})

'This is the third response from llm'

In [127]:
template_1 = PromptTemplate(
    template="Write a joke on {topic}",
    input_variables=['topic']
)

In [128]:
template_2 = PromptTemplate(
    template="Write explanation of this joke:\n{response}",
    input_variables=['response']
)

In [129]:
joke_llm = LLM()

LLM created


In [130]:
parser = StringOutputParser()

In [131]:
generate_joke_chain = RunnableChain([template_1, llm])

In [132]:
explain_joke_chain = RunnableChain([template_2, llm, parser])

In [133]:
final_chain = RunnableChain([generate_joke_chain, explain_joke_chain])

In [134]:
final_chain.invoke({'topic': 'pakistan'})

'This is the second response from llm'